In [1]:
from unittest import result

from conllu_test import check_conllu_for_conditions2, head_greater_than_id, head_smaller_than_id, has_son, no_deprel, file_to_conllu,merge_evals, conllu_merger, check_conllu_for_conditions_v3, no_weak_pronoun, has_sons, next_head, prev_head, has_no_son, check_conds, feats_include, merge_evals
import os
from pathlib import Path
import datetime
import json
from pathlib import Path
import pandas as pd
import time

from conllu_test import parent_is
from analysis_main import conllu_file_stats


In [2]:
# #FUNCTION 1: CHECK ONE FILE FOR SEVERAL CONDITIONS, return 1 dict of results
# # conditions are stored in a dictionary of conditions, with a name for the condition as key
# def eval_conds(conds_list, file):
#     results={"file": file }
#     for name, condition in conds_list.items():
#         results.update( {name :check_conllu_for_conditions2(file, condition, printf=False)} )
#     print(results)
#     return results

# FUNCTION 1 :input dict of conditions
# max ex refers to the maximum of examples it will include in the summary
#output: list of condition dictionaries, ready to json load
def eval_conds(d_conds, file, max_ex=100, custom_id= False):
    list=[]
    t0 = time.time()
    for name, cond in d_conds.items():
        t1=time.time()
        print(f"\tEvaluating {name} on {file}")
        results={"filename": file, "eval_series":0, "cond_name": name,  "matches":check_conllu_for_conditions2(file, cond, printf=False), "cond_content": cond, "examples":check_conllu_for_conditions_v3(file, cond, max_l=max_ex, custom_id=custom_id), "eval_date": datetime.datetime.now()}
        for key in results:
            results[key] = str(results[key])
        list.append(results)
        print(f"\t \tEvaluated {name} on {file}, it took {time.time()-t1}")
    print(f"done with list, it took {time.time()-t0}")
    return list


# FUNCITON 2: input: list of result dictionaries + name of the file the results should be printed in
# checks that the keys are the ones htat shoud be too
# If return_res is false, the dict gets printed in a json file, else it is returned
def print_eval_log(jlist, outfile, return_res=True):
    # check if the folder exists, else create it
    Path("eval/").mkdir(parents=True, exist_ok=True)

    outpath = "eval/"+outfile
    standard_dict = {"filename":0, "cond_name":"", "cond_content":0, "matches":0, "eval_date":0 , "examples": [], "eval_series":0, "cond_nr": 0}
    for result in jlist:
        #print("this is going ot be my dict: \n", result)
        dres= dict(result)
        if dres.keys() != standard_dict.keys():
            print(f"error keys dont match:\n {dres.keys()}\n vs. \n{standard_dict.keys()}")
            return 0
    if not return_res:
        with open(outpath, "w") as f:
            json.dump(jlist, f)
    else:
        return jlist

# list of condition lists comes in,
# file list comes in
# evaluates conditions on ht efiles and prints the result in  outfile.tsv, can be opened with libre office
def eval_pipeline(list_of_eval_sets, file_list, outfile):
    t0=time.time()
    res_col=[]
    conds_count=0
    for file in file_list:
        for num, eval_series in enumerate(list_of_eval_sets):
            print(f"Evaluating list {num+1} of {len(list_of_eval_sets)} on {file}")
            res_l=eval_conds(eval_series, file, max_ex=100, custom_id=True)
            for result in res_l:
                #adding a eval series number to the results from the same list of conditions (like nsubj list)
                result.update({"eval_series": num})
                result.update({"cond_nr": conds_count})
                res_col.append(result)
                conds_count+=1
    #print("this is the loop output:", res_col)
    evals=    json.dumps(print_eval_log(res_col, outfile, return_res=True))
    print("***SUCCESSFUL RUN***")
    print(f" {conds_count} evaluations done in {time.time()-t0}, av. time {(time.time()-t0)/conds_count} ")
    please=json.loads(evals)
    with open("eval/"+outfile, "w") as f:
        df=pd.DataFrame(please)
        df.to_csv(f, sep="\t", index=False)

    return 0

def merge_evals(condict1, condict2):
    mega_dict= {}
    for name1,cond1 in condict1.items():
        for name2, cond2 in condict2.items():
            l=[]
            for c1 in cond1:
                for c2 in cond2:
                    # unir condicions
                    z = c1 | c2
                    #print(z)
                    #si hi ha condicions incompatibles s hi afegeix una condicio impossible
                    for intersecting_keys in set(c1.keys()).intersection(set(c2.keys())):
                        if id(intersecting_keys) == id(has_sons):
                            #print(intersecting_keys, c1[intersecting_keys])
                            #print(intersecting_keys, c2[intersecting_keys])
                            z.update({has_sons : c1[intersecting_keys] + c2[intersecting_keys]})

                        if c1[intersecting_keys] != c2[intersecting_keys] and id(intersecting_keys) != id(has_sons):
                            #print("****non compatible conditions\n")
                            #print("condition 1\n:", c1[intersecting_keys])
                            #print("condition 2\n:",c2[intersecting_keys])
                            z = {"upos":"impossible"}
                    l.append(z)
            mega_dict.update({name1 + "__und__" + name2 : l})

    for cross_name, cross_cond in mega_dict.items():
        print(cross_name, "::::", cross_cond)
    return mega_dict


#computes for each file all of the freqs of the different values it might have
# but it allows us to include a condition (since i  want to get rid of conditions as lists of dictionaries, since i only use them with one dictionary, it only works on the first element of the list :))









#returns a list of all the values for the conllu feateures present in the corpus
def conllu_possible_values():
    good_conllu_keys=["upos", "xpos", "feats", "deprel"]

    d_categories={}
    for cat in good_conllu_keys:
        d_categories[cat]=conllu_file_freqs(tv3_corpus, cat, rel=True)
        d_categories[cat].update(conllu_file_freqs(KI_corpus, cat, rel=True))
        del d_categories[cat]["filename"]
        del d_categories[cat]["category"]
        del d_categories[cat]["rel"]
        d_categories[cat]= list(d_categories[cat].keys())
        print(cat, d_categories[cat], "\n\n")
    return d_categories

In [6]:
# list of all the values in the conllu file generated w the function conllu_possible_values()
deprel_list = ['det', 'nsubj', 'amod', 'case', 'obj', 'flat', 'ROOT', 'mark', 'obl', 'compound', 'cc', 'conj', 'punct', 'appos', 'nmod', 'aux', 'ccomp', 'advmod', 'fixed', 'acl', 'nummod', 'cop', 'advcl', 'expl:pass', 'xcomp', 'csubj', 'iobj', 'parataxis', 'dep']
upos_list = ['DET', 'NOUN', 'ADJ', 'ADP', 'PROPN', 'VERB', 'SCONJ', 'NUM', 'CCONJ', 'PUNCT', 'AUX', 'SYM', 'PRON', 'ADV', 'INTJ', 'SPACE', 'PART']

#EVAL SET 1
# presencia /absencia de subj i obj
conds_verb = [{"upos": "VERB"}]
conds_verb_nosubj = [{"upos": "VERB", has_no_son: {"deprel": "nsubj"}}]
conds_verb_wsubj = [{"upos": "VERB", has_sons: [{"deprel": "nsubj"}]} ]
conds_verb_wsubj_true = [{"upos": "VERB", has_sons: [{"deprel": "nsubj", has_no_son: {"deprel": "cop"} }]}]
conds_verb_wsubj_fake = [{"upos": "VERB", has_sons: [{"deprel": "nsubj", has_son: {"deprel": "cop"} }]}]

list_v_subj_basic = {"verb":conds_verb, "verb_no_nsubj": conds_verb_nosubj,"verb_w_nsubj":conds_verb_wsubj}

eval_fake_bsubj = [list_v_subj_basic, {"nsubj_fake": conds_verb_wsubj_fake, "nsubj_true": conds_verb_wsubj_true}]

conds_v_obj_good = [{"upos": "VERB", has_son: {"deprel": "obj", "lemma": no_weak_pronoun, has_no_son: {"upos": "ADP"}}  }]
conds_verb_no_good_obj = [{"upos": "VERB", has_no_son: {"deprel": "obj", "lemma": no_weak_pronoun, has_no_son: {"upos": "ADP"}} }]
list_v_obj_basic = {"verb":conds_verb, "verb_w_obj":conds_v_obj_good, "verb_no_obj":conds_verb_no_good_obj}

list_v_subj_x_v_obj= merge_evals(list_v_subj_basic, list_v_obj_basic)
eval_set_1 = [list_v_subj_basic, list_v_obj_basic, list_v_subj_x_v_obj]

# eval set 1a
# inclou iobj
conds_verb_iobj = [{"upos": "VERB", has_sons : [{"deprel": "iobj"}]}]
conds_verb_no_iobj = [{"upos": "VERB", has_no_son : {"deprel": "iobj"}}]

list_iobj_basic = {"verb":conds_verb, "v_w_iobj": conds_verb_iobj, "v_no_iobj": conds_verb_no_iobj}
eval_set_1a = eval_set_1 + [list_iobj_basic] + [merge_evals(list_iobj_basic, list_v_subj_x_v_obj)]

# Eval set 1b
# presencia i absencia de subj/obj pero excloent els prons rels (que? qui?)
# discerning rel. clauses

conds_verb_wsubj_relp= [{"upos": "VERB", has_sons: [{"deprel": "nsubj", feats_include: {"PronType": "Rel"} } ] }]
conds_verb_wsubj_norelp = [{"upos": "VERB", has_sons: [{"deprel": "nsubj"}], has_no_son: {"deprel": "nsubj", feats_include: {"PronType": "Rel"} }}]
list_v_nsubj_relp = {"v_nsubj_relp": conds_verb_wsubj_relp, "v_nsubj_norelp": conds_verb_wsubj_norelp}

conds_v_obj_good_relp = [{"upos": "VERB", has_son: {"deprel": "obj", "lemma": no_weak_pronoun, has_no_son: {"upos": "ADP"}, feats_include: {"PronType": "Rel"} }  }]
conds_v_obj_good_norelp = [{"upos": "VERB", has_son: {"deprel": "obj", "lemma": no_weak_pronoun, has_no_son: {"upos": "ADP"}}, has_no_son: {"deprel": "obj", "lemma": no_weak_pronoun, has_no_son: {"upos": "ADP"}, feats_include: {"PronType": "Rel"} }}]
list_v_obj_relp = {"v_obj_relp": conds_v_obj_good_relp, "v_obj_norelp": conds_v_obj_good_norelp}

list_v_obj_basic = {"verb":conds_verb, "verb_w_obj":conds_v_obj_good, "verb_no_obj":conds_verb_no_good_obj}


eval_set_1b = [list_v_subj_basic, merge_evals(list_v_subj_basic, list_v_nsubj_relp)] + [list_v_obj_basic, merge_evals(list_v_obj_basic, list_v_obj_relp)]

# EVAL SET 2
# other approach to checking the same: SV vs VS
# for the numbers to match we must include SVS and OVO #TODO: check if the numbers actually match

conds_verb_wsubj_SV = [{"upos": "VERB", has_sons: [{"deprel": "nsubj", "head": head_greater_than_id}] }]
conds_verb_wsubj_VS = [{"upos": "VERB", has_sons: [{"deprel": "nsubj", "head": head_smaller_than_id}] }]
conds_verb_wsubj_VSV = [{"upos": "VERB", has_sons: [{"deprel": "nsubj", "head": head_greater_than_id},{"deprel": "nsubj", "head": head_smaller_than_id}]}]

list_v_nsubj_order= {"v_SV":conds_verb_wsubj_SV, "v_VS":conds_verb_wsubj_VS, "v_VSV":conds_verb_wsubj_VSV}

conds_verb_wobj_OV = [{"upos": "VERB", has_sons: [{"deprel": "obj", "lemma": no_weak_pronoun, has_no_son: {"upos": "ADP"}, "head": head_greater_than_id}] }]
conds_verb_wobj_VO = [{"upos": "VERB", has_sons: [{"deprel": "obj", "lemma": no_weak_pronoun, has_no_son: {"upos": "ADP"}, "head": head_smaller_than_id}]}]
conds_verb_wobj_OVO = [ {"upos": "VERB", "head": head_greater_than_id, has_sons: [ {"deprel": "obj", "lemma": no_weak_pronoun, has_no_son: {"upos": "ADP"}} , {"deprel": "obj", "lemma": no_weak_pronoun, has_no_son: {"upos": "ADP"}, "head": head_smaller_than_id} ] } ]
list_v_obj_order = {"v_OV": conds_verb_wobj_OV, "v_VO": conds_verb_wobj_VO, "v_OVO": conds_verb_wobj_OVO}

eval_set_2 = eval_set_1 + [list_v_nsubj_order, list_v_obj_order]
eval_set_2a = eval_set_1b + [list_v_nsubj_order, list_v_obj_order] + [merge_evals(list_v_nsubj_order, list_v_obj_order)]

# eval set 2b
# same as 2, regarding order of the arguments respect to the verb
# but taking into account rel.pronouns
list_v_nsubj_order_relp = merge_evals(list_v_nsubj_order, list_v_nsubj_relp)
list_v_obj_order_relp = merge_evals(list_v_obj_order, list_v_obj_relp)
eval_set_2b = eval_set_2a + [list_v_nsubj_order_relp] + [list_v_obj_order_relp]


eval_set_2z= eval_set_1 + [list_iobj_basic] + [merge_evals(list_iobj_basic, list_v_subj_x_v_obj)] + eval_set_1b + [list_v_nsubj_order, list_v_obj_order] + [merge_evals(list_v_nsubj_order, list_v_obj_order)] + [list_v_nsubj_order_relp] + [list_v_obj_order_relp]

# EVAL SET #3
# v has a nsubj or obj, plus dissection of all upos of subject
conds_verb = [{"upos": "VERB"}]
conds_verb_wsubj = [{"upos": "VERB", has_sons: [{"deprel": "nsubj"}]}]

list_v_subj = {"verb":conds_verb, "verb_w_subj":conds_verb_wsubj}
list_v_nsubj_types={f"v_w_nsubj_{cat}": [{"upos": "VERB", has_sons: [{"deprel": "nsubj", "upos": cat}] }] for cat in upos_list}
list_v_nsubj_types.update(list_v_subj)


list_v_obj = {"verb":conds_verb, "verb_w_obj":conds_v_obj_good}
list_v_obj_types= {f"v_w_obj_{cat}": [{"upos": "VERB", has_sons: [{"deprel": "obj", "upos":cat, "lemma": no_weak_pronoun, has_no_son: {"upos": "ADP"}}]}] for cat in upos_list}
list_v_obj_types.update(list_v_obj)

eval_set_types = [list_v_nsubj_types, list_v_obj_types, merge_evals(list_v_nsubj_types, list_v_obj_types)]

#EVAL SET #4: reserved for other arguments, difficult to measure though, as they are systematically not well recognised -> ARE THEY?
# v with presence of sconj
conds_verb_wsconj= [{"upos": "VERB", has_sons: [{"upos": "SCONJ"}]}]
list_v_sconj = {"v_w_sconj": conds_verb_wsconj}



# EVAL SET #5 : adj and nouns
#
conds_adj = [{"upos": "ADJ"}]
conds_n_wadj = [{"upos": "NOUN", has_son: {"upos": "ADJ"}}]
conds_adj_w_n = [{"upos": "ADJ", parent_is: {"upos": "NOUN"}}]
conds_adj_nom = [{"upos": "ADJ", "head": head_greater_than_id, parent_is: {"upos": "NOUN"}}]
conds_nom_adj = [{"upos": "ADJ", "head": head_smaller_than_id, parent_is: {"upos": "NOUN"}}]
list_n_wadj_functions= {f"adj_function_{cat}":[{"upos": "NOUN", has_son: {"upos": "ADJ", "deprel": cat}}] for cat in deprel_list}

list_nadj = {"adj": conds_adj, "adj_wn": conds_adj_w_n,"AdjN": conds_adj_nom, "NAdj": conds_nom_adj}

eval_set_5 = [list_nadj, list_n_wadj_functions]



# EVAL SET #7
# analysing clauses are just more complex in one or the other (thereby having more subordinate clauses as arguments, sharing more arguments etc) -> still need to develop a test for that
# specially the xcomp should be a test for that, as the verb shares the subj w the second one
# also a test for
# core arguments:
conds_verb = [{"upos": "VERB"}]
conds_v_csubj = [{"upos": "VERB", has_sons: [{"deprel": "csubj"}]}]
conds_v_ccomp = [{"upos": "VERB", has_sons: [{"deprel": "ccomp"}]}]
conds_v_xcomp = [{"upos": "VERB", has_sons: [{"deprel": "xcomp"}]}]
conds_v_aux = [{"upos": "VERB", has_sons: [{"deprel": "aux"}]}]

#computing subordination levels
conds_v_sub0 = [{ "upos": "VERB", has_no_son : {"upos": "VERB"} }]
conds_v_sub1 = [{ "upos": "VERB", has_sons: [{ "upos": "VERB", has_no_son : {"upos": "VERB"} }] }]
conds_v_sub2 = [{ "upos": "VERB", has_sons: [{ "upos": "VERB", has_sons: [{"upos": "VERB", has_no_son : {"upos": "VERB"} }] }] }]
conds_v_sub3 = [{ "upos": "VERB", has_sons: [{"upos": "VERB", has_sons: [{"upos": "VERB", has_sons: [{"upos": "VERB", has_no_son : {"upos": "VERB"} }] }] }] }]
conds_v_sub4_or_more = [{ "upos": "VERB", has_sons: [{"upos": "VERB", has_sons: [{"upos": "VERB", has_sons: [{"upos": "VERB", has_sons: [{"upos": "VERB"}] }] }] }] }]

list_v_nested_v = {"v_0_verb_arguments": conds_v_sub0, "v_lvl_sub_1": conds_v_sub1, "v_lvl_sub_2": conds_v_sub2, "v_lvl_sub_3": conds_v_sub3, "v_lvl_4plus_sub": conds_v_sub4_or_more }

conds_aux = [{"upos": "AUX"}]
conds_aux_csubj = [{"upos": "AUX", has_sons: [{"deprel": "csubj"}]}]
conds_aux_ccomp = [{"upos": "AUX", has_sons: [{"deprel": "ccomp"}]}]
conds_aux_xcomp = [{"upos": "AUX", has_sons: [{"deprel": "xcomp"}]}]

list_NS = {"verb": conds_verb, "v_w_csubj": conds_v_csubj, "v_w_ccomp": conds_v_ccomp, "v_w_xcomp": conds_v_xcomp, "aux": conds_aux, "aux_csubj": conds_aux_csubj,"aux_ccomp": conds_aux_ccomp,"aux_xcomp": conds_aux_xcomp}
eval_set_7 = [list_NS]

#corpus
tv3_corpus="a1tv3_corpus.conllu"
KI_corpus="a1KI_corpus.conllu"

tv3_small= "tv3_corpus_10_russia.conllu"
KI_small= "KI_corpus_10_russia_gemm.conllu"
tv3_mid= "a1tv3_corpus_65_habitatge.conllu"
KI_mid= "a1KI_corpus_65_habitatge_gemm.conllu"

#some handy corpus lists:
file_list=[tv3_corpus, KI_corpus]
small_file_list=[tv3_small, KI_small]
mid_file_list=[tv3_mid, KI_mid]
mini_eval=[list_v_nsubj_order]
KI_corpus_list= ['a1KI_corpus_65_educacio_gemm.conllu', 'a1KI_corpus_65_musica_gemm.conllu', 'a1KI_corpus_65_migracions_gemm.conllu', 'a1KI_corpus_65_salut_gemm.conllu', 'a1KI_corpus_65_erc_gemm.conllu', 'a1KI_corpus_65_estatsunits_gemm.conllu', 'a1KI_corpus_65_meteorologia_gemm.conllu', 'a1KI_corpus_65_habitatge_gemm.conllu', 'a1KI_corpus_65_incendis_gemm.conllu', 'a1KI_corpus_65_psoe_gemm.conllu', 'a1KI_corpus_8_russia_gemm.conllu']
tv3_corpus_list= ['a1tv3_corpus_65_educacio.conllu', 'a1tv3_corpus_65_musica.conllu', 'a1tv3_corpus_65_migracions.conllu', 'a1tv3_corpus_65_salut.conllu', 'a1tv3_corpus_65_erc.conllu', 'a1tv3_corpus_65_estatsunits.conllu', 'a1tv3_corpus_65_meteorologia.conllu', 'a1tv3_corpus_65_habitatge.conllu', 'a1tv3_corpus_65_incendis.conllu', 'a1tv3_corpus_65_psoe.conllu', 'a1tv3_corpus_8_russia.conllu']
total_file_list= file_list + tv3_corpus_list + KI_corpus_list
eval_arg_order = [list_v_nsubj_order, list_v_obj_order, merge_evals(list_v_nsubj_order, list_v_obj_order)]



verb__und__verb :::: [{'upos': 'VERB'}]
verb__und__verb_w_obj :::: [{'upos': 'VERB', <function has_son at 0x7fa0eb79aca0>: {'deprel': 'obj', 'lemma': <function no_weak_pronoun at 0x7fa0eb79b240>, <function has_no_son at 0x7fa0eb79af20>: {'upos': 'ADP'}}}]
verb__und__verb_no_obj :::: [{'upos': 'VERB', <function has_no_son at 0x7fa0eb79af20>: {'deprel': 'obj', 'lemma': <function no_weak_pronoun at 0x7fa0eb79b240>, <function has_no_son at 0x7fa0eb79af20>: {'upos': 'ADP'}}}]
verb_no_nsubj__und__verb :::: [{'upos': 'VERB', <function has_no_son at 0x7fa0eb79af20>: {'deprel': 'nsubj'}}]
verb_no_nsubj__und__verb_w_obj :::: [{'upos': 'VERB', <function has_no_son at 0x7fa0eb79af20>: {'deprel': 'nsubj'}, <function has_son at 0x7fa0eb79aca0>: {'deprel': 'obj', 'lemma': <function no_weak_pronoun at 0x7fa0eb79b240>, <function has_no_son at 0x7fa0eb79af20>: {'upos': 'ADP'}}}]
verb_no_nsubj__und__verb_no_obj :::: [{'upos': 'impossible'}]
verb_w_nsubj__und__verb :::: [{'upos': 'VERB', <function has_son

In [11]:
conds_n_acl = [{"upos": "NOUN", "deprel": "acl"}]
test_list = {"noun_und_acl": conds_n_acl}
test_eval = [test_list]

#file_freqs_eval([tv3_corpus, KI_corpus])
#checking if the new exclusive conditions work
eval_pipeline(test_eval, mid_file_list, "noun_und_acl.csv")
#on times:
# each cond takes 6-8 sec in the big list

Evaluating list 1 of 1 on a1tv3_corpus_65_habitatge.conllu
	Evaluating noun_und_acl on a1tv3_corpus_65_habitatge.conllu
	 	Evaluated noun_und_acl on a1tv3_corpus_65_habitatge.conllu, it took 1.622464656829834
done with list, it took 1.622659683227539
Evaluating list 1 of 1 on a1KI_corpus_65_habitatge_gemm.conllu
	Evaluating noun_und_acl on a1KI_corpus_65_habitatge_gemm.conllu
	 	Evaluated noun_und_acl on a1KI_corpus_65_habitatge_gemm.conllu, it took 1.5469675064086914
done with list, it took 1.5471854209899902
***SUCCESSFUL RUN***
 2 evaluations done in 3.1758344173431396, av. time 1.5879236459732056 


0

In [4]:
     #file_freqs_eval_filtered_by_conds(mid_file_list, eval_set_1b, prefix = "SVO_basic_relpsubj_test")
#conds_dummy= [{feats_include: {"Gender":"Masc", "Number":"Sing"}}]
#list_dummy= {"relpron" : conds_dummy}

#eval_conds(list_dummy, tv3_small, 100, True)
conllu_file_stats(tv3_corpus)

{'words': 241620,
 'sentences': 8948,
 'av. sentence length': 27.00268216361198,
 'av. tree heigth': 4.900648189539562,
 'max tree heigth': 15,
 'tree height / sent length': 0.2022682266616944}

In [ ]:
conllu_file_stats(KI_corpus)

In [ ]:
small_eval_set=[list_nsubj]

eval_set=[list_nsubj]
eval_set_arg_structure= [merge_evals(list_v_subj, merge_evals(list_v_obj, list_v_sconj))]+[{"verb":conds_verb}]
full_eval_set=[list_nsubj, list_nadj] + eval_set_arg_structure
small_file_list=[tv3_small, KI_small]
file_list=[tv3_corpus, KI_corpus]

outfile="arg-smalltest.csv"

l_c1={"c1": conds_verb_wsubj_noun}
l_c2={"c2": conds_verb_wobj_noun}

eval_pipeline(eval_set_arg_structure,small_file_list, outfile)


In [3]:
#CODEBLOCK TO MERGE ALL MY FILES INTO ONE

l=65
tema= "salut"
n_model= "gemm"
llista_temes_sencera= {"educacio":269, "musica":234, "migracions":136, "salut":550, "erc":123, "estatsunits": 306, "meteorologia":1280,"habitatge":125, "incendis":117,  "psoe":110}
plus={"russia":144, "israel": 101,}

KI_file=f"a1KI_corpus_{l}_{tema}_{n_model}.json"
KI_conllu= KI_file.strip(".json")+".conllu"
tv3_file= KI_file.replace("KI","tv3")
tv3_conllu= KI_conllu.replace("KI","tv3")
tv3_conllu = tv3_conllu.replace("_"+n_model,"")


KI_conllu_list = [f"a1KI_corpus_65_{tema}_gemm.conllu" for tema in llista_temes_sencera.keys()]
tv3_conllu_list= [f"a1tv3_corpus_65_{tema}.conllu" for tema in llista_temes_sencera.keys()]

KI_conllu_list.append("a1KI_corpus_8_russia_gemm.conllu")
#KI_conllu_list.append("KI_corpus_28_israel_gemm.conllu")
#tv3_conllu_list.append("tv3_corpus_28_israel.conllu")
tv3_conllu_list.append("a1tv3_corpus_8_russia.conllu")

conllu_merger(KI_conllu_list, "a1KI_corpus.conllu")
conllu_merger(tv3_conllu_list, "a1tv3_corpus.conllu")



In [6]:
#print(tv3_conllu_list)
#conllu_file_freqs(tv3_corpus, "upos", rel=True)
#conllu_file_freqs(KI_corpus, "upos", rel=True)
#file_list=[tv3_corpus, KI_corpus]+KI_conllu_list+tv3_conllu_list

conllu_file_stats(tv3)

#file_freqs_eval(file_list)

['a1tv3_corpus_65_educacio.conllu', 'a1tv3_corpus_65_musica.conllu', 'a1tv3_corpus_65_migracions.conllu', 'a1tv3_corpus_65_salut.conllu', 'a1tv3_corpus_65_erc.conllu', 'a1tv3_corpus_65_estatsunits.conllu', 'a1tv3_corpus_65_meteorologia.conllu', 'a1tv3_corpus_65_habitatge.conllu', 'a1tv3_corpus_65_incendis.conllu', 'a1tv3_corpus_65_psoe.conllu', 'a1tv3_corpus_8_russia.conllu']


In [ ]:
tv3_corpus="KI_corpus.conllu"
KI_corpus="tv3_corpus.conllu"
#merge all files that have 65 in the name
#conllu_merger(tv3_conllu_list, mega_tv3_file)
#conllu_merger(KI_conllu_list, mega_KI_file)

#file_to_conllu(mega_tv3_file)


In [ ]:
conllu_file_stats(KI_corpus)

In [ ]:

conllu_file_stats(tv3_corpus)


In [64]:
#concerning the question about the vocabulary diversity in adj that come before nouns: how does the position affect vocabulary? Is for the KI AdjN reserved for special (ehem english) adjectives?
print(conllu_file_freqs(tv3_corpus, "upos", conds=conds_adj , name_cond="adj" , rel=False))
for keys, values in list_nadj.items():
    d=conllu_file_freqs(tv3_corpus, "lemma", conds=values , name_cond= keys, rel=False)
    print(keys, len(d)-4)

print(conllu_file_freqs(KI_corpus, "upos", conds=conds_adj , name_cond="adj" , rel=False))
for keys, values in list_nadj.items():
    d=conllu_file_freqs(KI_corpus, "lemma", conds=values , name_cond= keys, rel=False)
    print(keys, len(d)-4)

KeyboardInterrupt: 

In [ ]:
eval_conds(conditions_list_verbs, mega_tv3_file)

In [ ]:
eval_conds(conditions_list_verbs, mega_KI_file)

In [ ]:
results_tv3= eval_conds(conditions_list_verbs, mega_tv3_file)

In [ ]:
results_KI= eval_conds(conditions_list_verbs, mega_KI_file)

In [ ]:
for keys, values in results_KI.items():
    if isinstance(results_KI[keys], int):
        print(keys, values/results_KI["verb"])


In [ ]:
for keys, values in results_tv3.items():
    if isinstance(results_tv3[keys], int):
        print(keys, values/results_tv3["verb"])

In [ ]:
conds_verb_wobj = [{"upos": "VERB", has_son: {"deprel": "obj"}}]
conds_verb_wobj_noun = [{"upos": "VERB", has_son: {"deprel": "obj", "upos": "NOUN"}}]
conds_verb_wobj_propn = [{"upos": "VERB", has_son: {"deprel": "obj", "upos": "PROPN"}}]
conds_verb_wobj_pron = [{"upos": "VERB", has_son: {"deprel": "obj", "upos": "PRON"}}]


conds_verb_wsubj_wobj = [{"upos": "VERB", has_son: {"deprel":"nsubj"}, has_son: {"deprel": "obj"}}]
conds_verb_wsubj_wobj_noun = [{"upos": "VERB", has_son: {"deprel":"nsubj"}, has_son: {"deprel": "obj", "upos": "NOUN"}}]
conds_verb_wsubj_wobj_propn = [{"upos": "VERB", has_son: {"deprel": "obj", "upos": "PROPN"}, has_son: {"deprel":"nsubj"}}]

list_v_obj={"verb_obj":conds_verb_wobj, "v_obj_n": conds_verb_wobj_noun, "v_obj_propn": conds_verb_wobj_propn, "v_obj_pron": conds_verb_wobj_pron}